In [17]:
import os
from typing import Annotated, List, TypedDict, Optional, Union
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI

In [18]:
# The specialized framework for deep reasoning agents
from deepagents import create_deep_agent

In [19]:
load_dotenv()

True

In [20]:
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    paper_path: Optional[str]
    scientific_logic: Optional[str]
    implementation_code: Optional[str]

print("Environment and State initialized.")

Environment and State initialized.


In [21]:
llm= ChatGoogleGenerativeAI(
    model= "gemini-2.0-flash",
    temperature= 0
)

In [22]:
research_agent= create_deep_agent(
    model= llm,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your job is to read research papers, extract the mathematical core, "
        "and explain the architecture in plain pseudocode. "
        "Use your planning tools to break down the paper analysis step-by-step."
    )
)
print("Deep Research Agent created.")

Deep Research Agent created.


In [23]:
import arxiv
import fitz  
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

In [24]:
# 1. arXiv Search and Download Tool
@tool
def download_research_paper(query: str)-> str:
    """
    Searches arXiv for a paper, downloads the PDF to the '../data/' folder, 
    and returns the local path to the file.
    """
    download_path= "../data/"
    os.makedirs(download_path, exist_ok= True)

    search= arxiv.Search(query= query, max_results= 1, sort_by= arxiv.SortCriterion.Relevance)
    paper= next(search.results())

    filename = f"{paper.title.replace(' ', '_').replace(':', '')}.pdf"
    full_path = paper.download_pdf(dirpath=download_path, filename=filename)
    
    return f"Paper downloaded to: {full_path}"

In [25]:
# 2. PDF Reading Tool
@tool
def read_pdf_content(file_path: str) -> str:
    """
    Opens a PDF file and extracts the text content. 
    Useful for reading downloaded research papers.
    """
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [26]:
# 3. Web Search Tool (Tavily)
web_search = TavilySearchResults(k=3)

In [27]:
# Combine all tools
tools = [download_research_paper, read_pdf_content, web_search]
print("ArXiv, PDF, and Tavily tools defined.")

ArXiv, PDF, and Tavily tools defined.


In [28]:
# 1. Re-initialize the Researcher Agent with tools
# We give it our 'tools' list so it can search, download, and read.
researcher_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your mission is to find research papers and extract their implementation core. "
        "Step 1: Use 'download_research_paper' to get the PDF. "
        "Step 2: Use 'read_pdf_content' to understand the math and logic. "
        "Step 3: Output a structured explanation of the algorithm, variables, and pseudocode."
    )
)
print("Research Agent re-Linked with tools.")

Research Agent re-Linked with tools.


In [30]:
# trigger a "Deep" research session
inputs = {
    "messages": [
        ("user", "Research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.")
    ]
}

config = {"configurable": {"thread_id": "research-task-1"}}
print("Starting Deep Research. Watch the agent plan and execute...")

for chunk in research_agent.stream(inputs, config):
    print(chunk)

Starting Deep Research. Watch the agent plan and execute...
{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.', additional_kwargs={}, response_metadata={}, id='b1bd5af5-3a39-4e1c-80f6-68454f4dbed1')])}}
{'model': {'messages': [AIMessage(content='Okay, I will research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.', additional_kwargs={'function_call': {'name': 'write_todos', 'arguments': '{"todos": [{"content": "Research the QLoRA paper to understand the 4-bit NormalFloat quantization method.", "status": "in_progress"}, {"status": "pending", "content": "Explain the 4-bit NormalFloat quantization method based on the research."}]}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf6f7-6239-7cf3-a1ed-ed3723d52dd8-0', tool_calls=[{'name

In [31]:
# This agent takes the research findings and plans the Code structure.
designer_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt= (
        "You are the Architecture Designer. "
        "Your goal is to take research findings and create a high-level Design Spec. "
        "1. Define the Python classes and methods needed. "
        "2. Specify the exact mathematical formulas for each method. "
        "3. Decide on the libraries to use (e.g., torch, numpy). "
        "Output ONLY the design specification, not the actual code yet."
    )
)
print("Architecture Designer Agent created.")

Architecture Designer Agent created.


In [32]:
workflow= StateGraph(ResearchState)


# 1. Add Nodes (Link our agents to the graph)
workflow.add_node("researcher", lambda state: {"messages": [researcher_agent.invoke(state["messages"])]})
workflow.add_node("designer", lambda state: {"messages": [designer_agent.invoke(state["messages"])]})

# 2. Define the Flow (Researcher -> Designer)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "designer")
workflow.add_edge("designer", END)

# 3. Compile the Graph
app = workflow.compile()
print("Multi-Agent Workflow Compiled!")

Multi-Agent Workflow Compiled!
